In [1]:
import numpy as np
from scipy.linalg import hadamard

H4 = hadamard(8)

# Calculate H4 @ H4^T
H4_H4_T = H4 @ H4.T

# Calculate H4[:, :1] @ H4[:, :1]^T (outer product of the first column of H4 with itself)
H4_col1 = H4[:, :2]
H4_col1_H4_col1_T = H4_col1 @ H4_col1.T

H4_col1_H4_col1_T

array([[2, 0, 2, 0, 2, 0, 2, 0],
       [0, 2, 0, 2, 0, 2, 0, 2],
       [2, 0, 2, 0, 2, 0, 2, 0],
       [0, 2, 0, 2, 0, 2, 0, 2],
       [2, 0, 2, 0, 2, 0, 2, 0],
       [0, 2, 0, 2, 0, 2, 0, 2],
       [2, 0, 2, 0, 2, 0, 2, 0],
       [0, 2, 0, 2, 0, 2, 0, 2]])

In [2]:
H4[:,:2]

array([[ 1,  1],
       [ 1, -1],
       [ 1,  1],
       [ 1, -1],
       [ 1,  1],
       [ 1, -1],
       [ 1,  1],
       [ 1, -1]])

In [9]:
H4[:,[0,4]] @ H4[:,[0,4]].T

array([[2, 2, 2, 2, 0, 0, 0, 0],
       [2, 2, 2, 2, 0, 0, 0, 0],
       [2, 2, 2, 2, 0, 0, 0, 0],
       [2, 2, 2, 2, 0, 0, 0, 0],
       [0, 0, 0, 0, 2, 2, 2, 2],
       [0, 0, 0, 0, 2, 2, 2, 2],
       [0, 0, 0, 0, 2, 2, 2, 2],
       [0, 0, 0, 0, 2, 2, 2, 2]])

In [175]:
color4 = np.zeros((8,4))
color4[:3,0] = 1
color4[3:5,1] = 1
color4[5:7,2] = 1
color4[7:8,3] = 1
color4 = color4[:,:2]
color4 @ color4.T

array([[1., 1., 1., 0., 0., 0., 0., 0.],
       [1., 1., 1., 0., 0., 0., 0., 0.],
       [1., 1., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 1., 0., 0., 0.],
       [0., 0., 0., 1., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.]])

In [24]:
H4[:,0] @ H4[:,1]

0

In [61]:
import numpy as np

class MeshVars:
    def __init__(self, dimensions, points_per_dimension):
        self.d = dimensions
        self.ptsPerDim = points_per_dimension
        self.dmax = np.argmax(points_per_dimension)
        self.twoTod = 2 ** self.d
        self.logPts = np.floor(np.log2(points_per_dimension)).astype(int)
        self.RBorder = None
        self.PTbits = None


def int2bin(n, array_size):
    return format(n, f'0{array_size}b')


def bin2int(binary_str):
    if not binary_str:
        return 0
    return int(binary_str, 2)


def find_RBorder(mesh):
    cols = [0]
    for i in range(1, mesh.d + 1):
        next_cols = cols + [not col for col in cols]
        cols = next_cols
    
    reds = 0
    blacks = mesh.twoTod // 2
    mesh.RBorder = [''] * mesh.twoTod
    for i, col in enumerate(cols):
        destination = reds if col == 0 else blacks
        if col == 0:
            reds += 1
        else:
            blacks += 1
        mesh.RBorder[i] = int2bin(destination, mesh.d)


def hierOrderSetUp(mesh):
    mesh.RBorder = [''] * (mesh.twoTod * mesh.d)
    find_RBorder(mesh)
    mesh.PTbits = np.zeros((mesh.d, mesh.logPts[mesh.dmax]), dtype=str)


def hierOrderPoint(coord, mesh):
    logmax = mesh.logPts[mesh.dmax]
    location_str = ''
    for j in range(mesh.d):
        mesh.PTbits[j, :mesh.logPts[j]] = list(int2bin(coord[j], mesh.logPts[j]))
    
    for m in range(logmax, 0, -1):
        crossbits = ['0'] * mesh.d
        for j in range(mesh.d):
            if logmax - m < mesh.logPts[j]:
                crossbits[j] = mesh.PTbits[j, logmax - m]
        from_index = bin2int(''.join(crossbits)) * mesh.d
        to_index = from_index + sum(logmax - m < lp for lp in mesh.logPts)
        location_str += ''.join(mesh.RBorder[from_index:to_index])
        print(location_str)
    return bin2int(location_str)


def freeMeshVars(mesh):
    del mesh.ptsPerDim
    del mesh.logPts
    del mesh.RBorder
    del mesh.PTbits

def index2coord(index, mesh):
    coord = np.zeros(mesh.d, dtype=int)
    for j in range(mesh.d):
        div = np.prod(mesh.ptsPerDim[:j]) if j > 0 else 1
        coord[j] = (index // div) % mesh.ptsPerDim[j]
    return coord


def hierPerm(mesh, N):
    perm = np.zeros(N, dtype=int)
    for i in range(N):
        coord = index2coord(i, mesh)
        perm[i] = hierOrderPoint(coord, mesh)
    return perm


def hada_col_perm(N, Hpsize):
    Hperm = np.zeros(Hpsize, dtype=int)
    icur = 0
    newindices = 1
    step = N // 2
    while newindices < Hpsize:
        newindices *= 2
        for i in range(1, newindices, 2):
            if icur >= Hpsize:
                break
            Hperm[icur] = i * step
            icur += 1
        step //= 2
    return Hperm


def num1bits(a):
    return bin(a).count('1')


def Hada_element(i, j):
    bits = num1bits(i & j)
    return -1 if bits % 2 else 1


# Assuming a simplified use-case without the specifics of Chroma or layout sizes.
# This function needs more context about Chroma and Layout to be fully implemented.
def hier_perm_for_chroma(mesh, layout_sizes):
    Nx, Ny, Nz, Nt = layout_sizes
    N = Nx * Ny * Nz * Nt
    perm = np.zeros(N, dtype=int)
    for x in range(Nx):
        for y in range(Ny):
            for z in range(Nz):
                for t in range(Nt):
                    coord = [x, y, z, t]
                    i = x + y*Nx + z*Nx*Ny + t*Nx*Ny*Nz  # Simplified linear index calculation
                    perm[i] = hierOrderPoint(coord, mesh)
    return perm



In [33]:
dimensions = 4  # Example: 4-dimensional mesh
points_per_dimension = [2, 2, 2, 2]  # Example: 2 points per dimension
mesh = MeshVars(dimensions, points_per_dimension)
hierOrderSetUp(mesh)
coord = [1, 1, 1, 1]  # Example coordinate
print(hierOrderPoint(coord, mesh))
freeMeshVars(mesh)

0


In [62]:
dimensions = 4
points_per_dimension = [2, 2, 2, 2]  # Updated for a 4x4x4x4 grid
mesh = MeshVars(dimensions, points_per_dimension)
hierOrderSetUp(mesh)

# For a given index, find its coordinate and hierarchical order
# index = 5  # Example index
# coord = index2coord(index, mesh)
# print(f"Coordinate for index {index}: {coord}")
# print(f"Hierarchical order for index {index}: {hierOrderPoint(coord, mesh)}")

# Setup permutations for hierarchical ordering
N = np.prod(points_per_dimension)
perm = hierPerm(mesh, N)
print(f"Permutation for hierarchical ordering:\n", perm)  # Show first 10 for brevity

# Setup Hadamard column permutation
Hpsize = 8  # Example Hadamard vector size
Hperm = hada_col_perm(N, Hpsize)
print(f"Hadamard column permutation: {Hperm}")

RHS = np.zeros((len(Hperm), N))
for sample, h in enumerate(Hperm):
    for i in range(N):
        RHS[sample, i] = Hada_element(perm[i], h)
print(RHS)
RHS.shape

0000100010010001



1100010001011101



1010001000111011



0110111011110111



Permutation for hierarchical ordering:
 [ 2193     0     0     0 50269     0     0     0 41531     0     0     0
 28407     0     0     0]
Hadamard column permutation: [ 8  4 12  2  6 10 14  0]
[[ 1.  1.  1.  1. -1.  1.  1.  1. -1.  1.  1.  1.  1.  1.  1.  1.]
 [ 1.  1.  1.  1. -1.  1.  1.  1.  1.  1.  1.  1. -1.  1.  1.  1.]
 [ 1.  1.  1.  1.  1.  1.  1.  1. -1.  1.  1.  1. -1.  1.  1.  1.]
 [ 1.  1.  1.  1.  1.  1.  1.  1. -1.  1.  1.  1. -1.  1.  1.  1.]
 [ 1.  1.  1.  1. -1.  1.  1.  1. -1.  1.  1.  1.  1.  1.  1.  1.]
 [ 1.  1.  1.  1. -1.  1.  1.  1.  1.  1.  1.  1. -1.  1.  1.  1.]
 [ 1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.]
 [ 1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.]]


(8, 16)

In [54]:
mesh.PTbits

array([['1'],
       ['1'],
       ['1'],
       ['1']], dtype='<U1')

In [73]:
np.random.rand(5,5)

array([[0.74230083, 0.49607123, 0.44658698, 0.86185609, 0.75686061],
       [0.07973893, 0.14900821, 0.50008636, 0.84857802, 0.00753508],
       [0.82361944, 0.60516665, 0.8117063 , 0.1662096 , 0.70830452],
       [0.49915544, 0.21647981, 0.95870934, 0.34312924, 0.19011628],
       [0.58095426, 0.46980463, 0.32541998, 0.43342168, 0.19158113]])

In [139]:
import numpy as np
from scipy.sparse import rand
from scipy.sparse.linalg import inv

# Generate a random sparse matrix of size 5x5 with density 0.2
sparse_matrix = rand(5, 5, density=0.4, dtype=np.float64).toarray()
# sparse_matrix += np.random.rand(5,5) * 1e-10

# Calculate its inverse
inverse_matrix = np.linalg.inv(sparse_matrix)

# Convert the inverse matrix to a dense format for display
# dense_inverse_matrix = inverse_matrix.toarray()
np.set_printoptions(precision=3, suppress=True)
inverse_matrix#sparse_matrix.toarray(), dense_inverse_matrix

array([[  1.003,  31.59 ,  -0.619, -54.544,  -0.45 ],
       [  0.   , -32.884,   0.   ,  56.866,   0.   ],
       [  0.   ,   1.483,   0.   ,   0.   ,   0.   ],
       [  0.   ,  -0.131,   1.668,   0.   ,   0.   ],
       [  0.   ,   0.   ,   0.   ,   0.   ,   1.165]])

In [140]:
sparse_matrix

array([[0.997, 0.957, 0.   , 0.37 , 0.385],
       [0.   , 0.   , 0.674, 0.   , 0.   ],
       [0.   , 0.   , 0.053, 0.599, 0.   ],
       [0.   , 0.018, 0.39 , 0.   , 0.   ],
       [0.   , 0.   , 0.   , 0.   , 0.858]])

In [137]:
sparse_matrix[0,:]

array([0.   , 0.731, 0.518, 0.605, 0.874])

In [138]:
inverse_matrix[:,0]

array([-0.256,  1.368,  0.   ,  0.   , -0.   ])

In [144]:
sparse_matrix[0,:].reshape(-1,1) @ inverse_matrix[:,0].reshape(1,-1)

array([[1.   , 0.   , 0.   , 0.   , 0.   ],
       [0.959, 0.   , 0.   , 0.   , 0.   ],
       [0.   , 0.   , 0.   , 0.   , 0.   ],
       [0.371, 0.   , 0.   , 0.   , 0.   ],
       [0.386, 0.   , 0.   , 0.   , 0.   ]])

In [131]:
inverse_matrix[:,0]

array([ 0.   , -2.209, -0.   ,  0.   ,  2.664])

In [154]:
u = np.array([1,0.5,0.25,0.5,1])
v = np.array([1,1/2,1/3,1/2,1])
u.reshape(-1,1) @ v.reshape(1,-1)

array([[1.   , 0.5  , 0.333, 0.5  , 1.   ],
       [0.5  , 0.25 , 0.167, 0.25 , 0.5  ],
       [0.25 , 0.125, 0.083, 0.125, 0.25 ],
       [0.5  , 0.25 , 0.167, 0.25 , 0.5  ],
       [1.   , 0.5  , 0.333, 0.5  , 1.   ]])